<a href="https://colab.research.google.com/github/amrit2603/Gen-AI/blob/main/PEFT_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

dataset=load_dataset("SetFit/emotion")

README.md:   0%|          | 0.00/194 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/2.23M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/276k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/279k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [2]:
train_data=dataset["train"]
test_data=dataset["test"]

In [3]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
# Tokenization function
def tokenize_function(batch):
    return tokenizer(batch["text"],padding="max_length",truncation=True)

# Apply tokenization
train_dataset = train_data.map(tokenize_function, batched=True)
test_dataset = test_data.map(tokenize_function, batched=True)

# Convert to PyTorch format
train_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification

#load bert model
base_model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=6)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from peft import  get_peft_model, prepare_model_for_kbit_training, TaskType, LoraConfig

# Define LoRA Configuration
lora_config = LoraConfig(
    r=8,                 # Low-rank adaptation dimension , the more the value the better the performance but also more computation
    lora_alpha=32,       # Scaling factor
    lora_dropout=0.05,   # Dropout rate
    target_modules=["query", "value"]  # Apply LoRA to self-attention layers only , coz we are using bert which is transformer based model
)

# Prepare model for LoRA
base_model = prepare_model_for_kbit_training(base_model)

# Convert model into LoRA-enabled model
peft_model = get_peft_model(base_model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 294,912 || all params: 109,781,766 || trainable%: 0.2686


In [7]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU count: 1
GPU name: Tesla T4


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(  # configuration class that defines how training should happen
    output_dir="./model_checkpoints",   # Where to save model
    num_train_epochs=3,                 # Train for 3 epochs
    per_device_train_batch_size=16,     # 16 samples per GPU/CPU
    eval_strategy="epoch",              # Evaluate after every epoch
    save_strategy="epoch",              # Save model after each epoch
    logging_steps=10,                   # Log training metrics every 10 steps
    load_best_model_at_end=True,         # Automatically load best checkpoint
    fp16=False                            # Use mixed precision for faster training (if GPU supports it)
)

# A high-level class that automates training, evaluation, and saving models.
# It wraps around your model and dataset, handling:
# - Training loops
# - Evaluation during training
# - Model saving & checkpointing

# Create a data collator to handle dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=peft_model,          # LoRA fine-tuned model
    args=training_args,        # Training settings
    train_dataset=train_dataset,  # Training data
    eval_dataset=test_dataset,    # Test data
    data_collator=data_collator # Pass the data collator here instead of tokenizer directly
)

trainer.train()

[3000/3000 8:31:56, Epoch 3/3]
Epoch	Training Loss	Validation Loss
1	1.313000	1.165364
2	1.067100	1.015033
3	1.094000	0.963863


In [10]:
model = trainer.model
model.eval()

PeftModel(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, o

In [11]:
device = next(model.parameters()).device

device

device(type='cuda', index=0)

In [13]:
# Example: Tokenize a sample text to create 'inputs'
sample_text = "I am so happy today!"
inputs = tokenizer(sample_text, return_tensors="pt", padding="max_length", truncation=True)

new_inputs = {}

for k, v in inputs.items():
    new_inputs[k] = v.to(device)

inputs = new_inputs

# v.to(device) means moving each tensor to the device and k,v are key value pairs in the inputs dictionary
# k = 'input_ids', 'attention_mask', etc.
# v = the corresponding tensor

# 🌟🌟🌟🌟🌟
# here we are moving the inputs to the same device as model
# this is important for computation to happen on the same device
# training can happen on two devices cpu or gpu but inference should happen on the same device as model

In [14]:
import torch

model.eval()

device = next(model.parameters()).device

tests = [
    "I absolutely love this, best thing ever",
    "This is horrible, I hate it so much",
    "Worst product in the world",
    "Fantastic, exceeded all expectations"
]

inputs = tokenizer(
    tests,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# move inputs to same device as model
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
confidence, preds = torch.max(probs, dim=-1)

for text, pred, conf in zip(tests, preds, confidence):
    print(text)
    print(f" → class {pred.item()}, confidence {conf.item():.2f}")

I absolutely love this, best thing ever
 → class 5, confidence 0.24
This is horrible, I hate it so much
 → class 5, confidence 0.24
Worst product in the world
 → class 5, confidence 0.25
Fantastic, exceeded all expectations
 → class 5, confidence 0.22
